# 00. 가이드북과 1차 가공 데이터 검토

**새 모델 학습 전의 자료 이해 단계**입니다. 기존 실험은 `../archive/2026-09-29_tabular_v1/`에 보존했습니다.

먼저 [가이드북 상세 분석](../docs/guidebook_review.md)을 읽어 주세요. 이 노트북은 저장된 진단 요약으로 실제 파일의 구성·기간·라벨·중복을 확인합니다. 커널은 **Python (kamp-base)**, 실행 순서는 위에서 아래입니다. 재시작 후 중간 셀만 실행하면 앞 셀의 변수가 없어 오류가 날 수 있습니다.

타임스탬프가 있다는 사실만으로 LSTM이나 순수 비지도학습이 정답인 것은 아닙니다. 지금은 시간과 공정 문맥을 복구하고, 이후 같은 시간 검증에서 여러 접근을 비교할 준비를 합니다.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# 이 노트북 또는 프로젝트 루트에서 실행할 수 있습니다.
PROJECT = None
for base in [Path.cwd(), *Path.cwd().parents]:
    for candidate in (base, base / "1.injection-molding"):
        if (candidate / "scripts/review_sources.py").exists():
            PROJECT = candidate
            break
    if PROJECT is not None:
        break
if PROJECT is None:
    raise FileNotFoundError("현재 사출 프로젝트에서 노트북을 열어 주세요.")
RESULTS = PROJECT / "artifacts/source_review_v1"
print("프로젝트:", PROJECT)
print("Python:", sys.executable)

## 1. 진단을 다시 만들지 선택

기본값 `False`는 저장된 결과만 읽습니다. `True`는 원본 ZIP을 읽어 새로운 시각별 폴더에 진단을 생성합니다. 모델을 학습하지 않으며 ZIP을 수정하지 않습니다. 약 80만 행의 파일을 읽으므로 메모리 여유가 필요합니다.

In [ ]:
RUN_AUDIT = False
if RUN_AUDIT:
    from datetime import datetime
    RESULTS = PROJECT / "artifacts/source_review_runs" / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    subprocess.run([sys.executable, str(PROJECT / "scripts/review_sources.py"),
                    "--output", str(RESULTS)], check=True)
if not (RESULTS / "audit.json").exists():
    raise FileNotFoundError("sources에 ZIP을 배치하고 RUN_AUDIT=True로 실행하세요.")
audit = json.loads((RESULTS / "audit.json").read_text())
print("읽는 결과:", RESULTS)

## 2. 어떤 파일이 있는가

`labeled_data.csv`와 `unlabeled_data.csv`가 새 분석의 중심입니다. 나머지는 가공된 파생 자료이므로 8개를 단순 합치지 않습니다. 1차 가공 파일도 미가공 센서 파형과 같은 말은 아닙니다.

In [ ]:
inventory = pd.read_csv(RESULTS / "file_inventory.csv")
display(inventory[["file", "rows", "columns", "duplicate_rows_without_export_index", "unique_ids", "time_min", "time_max"]])

## 3. 양품/불량의 의미와 불량 사유

ID 연결 결과는 Y→0(양품), N→1(불량)입니다. `Reason`의 빈 값은 양품의 경우 불량 사유 없음과 맞물립니다. 모든 빈 값을 오류로 제거하지 않습니다. 아래는 같은 ID를 한 번만 센 집계이며 원본 삭제가 아닙니다.

In [ ]:
display(pd.DataFrame(audit["label_mapping"]))
reasons = pd.read_csv(RESULTS / "reason_counts.csv")
unique = reasons[(reasons["file"] == "labeled_data.csv") & (reasons["count_unit"] == "unique_id")].copy()
display(unique[["PART_NAME", "PassOrFail", "Reason", "rows"]])
print("전체 행:", audit["labeled_data.csv"]["PassOrFail"])
print("중복 ID 제외:", audit["labeled_data.csv"]["unique_id_label_counts_not_training_dedup"])

가이드북은 쇼트숏·플래시·휨 등 여러 현상을 설명하지만, 실제 사유는 가스·미성형·초기허용불량뿐입니다. 없는 불량 종류를 정답으로 만들어 학습하지 않습니다. 더 넓은 자료에서는 RG3 왼쪽의 불량도 있으므로 이전 부분집합의 관찰을 전체로 일반화할 수 없습니다.

## 4. 시간 범위와 기록 간격

같은 설비·부품에서 중복 시각을 제외한 기록 간격입니다. 생산 중단 기간도 간격에 포함됩니다. 시간 정보가 있어도 한 사이클 안의 고속 센서 파형이라는 뜻은 아닙니다.

In [ ]:
coverage = pd.read_csv(RESULTS / "product_coverage.csv")
focus = coverage[(coverage["EQUIP_CD"] == "S14") & coverage["PART_NAME"].str.contains("CN7|RG3")].copy()
display(focus)
intervals = pd.read_csv(RESULTS / "timestamp_intervals.csv")
display(intervals[(intervals["file"] == "labeled_data.csv") & (intervals["EQUIP_CD"] == "S14")][["PART_NAME", "unique_timestamps", "min_seconds", "p50_seconds", "p95_seconds", "max_seconds"]])

# 제공 범위 그림: 연속선은 관측 범위이며 모든 시각에 관측이 있다는 뜻이 아닙니다.
fig, ax = plt.subplots(figsize=(11, 4))
for i, (_, row) in enumerate(focus.iterrows()):
    part = row["PART_NAME"].strip()
    label = ("CN7" if "CN7" in part else "RG3") + " " + part[-2:] + " / " + row["file"].replace(".csv", "")
    ax.plot(pd.to_datetime([row["time_min"], row["time_max"]]), [i, i], marker="o", label=label)
ax.set_yticks(range(len(focus)))
ax.set_yticklabels([("CN7" if "CN7" in r.PART_NAME else "RG3") + " " + r.PART_NAME.strip()[-2:] + " / " + r.file.replace(".csv", "") for r in focus.itertuples()])
ax.set_title("S14 data coverage (gaps within each range are not shown)")
ax.grid(axis="x", alpha=.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 5. 날짜별 불량 기록

관측된 날짜의 기록 개수입니다. 공장 전체 생산량·불량률을 추정한 것이 아닙니다. 막대 아래의 정상/불량 기록 수를 보고, 나중에 시간 평가를 나눌 때 각 구간에 실제 불량이 있는지 확인해야 합니다.

In [ ]:
daily = pd.read_csv(RESULTS / "labeled_daily_counts.csv")
s14 = daily[daily["EQUIP_CD"] == "S14"].copy()
s14["product"] = s14["PART_NAME"].str.extract(r"(CN7|RG3)")
daily_counts = s14.groupby(["product", "date", "PassOrFail"])["unique_records"].sum().unstack(fill_value=0)
display(daily_counts)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for ax, product in zip(axes, ["CN7", "RG3"]):
    table = daily_counts.loc[product]
    ax.bar(table.index, table.get("N", pd.Series(0, index=table.index)))
    ax.set_title(product + " / labeled defect records by observed date")
    ax.set_ylabel("Defect records (deduplicated ID)")
    ax.tick_params(axis="x", rotation=60)
plt.tight_layout()
plt.show()

## 6. 미라벨 학습 전에 막아야 할 중복과 정보 유출

다른 ID라도 같은 이벤트일 수 있습니다. 라벨/미라벨 간 설비·부품·시각이 같은 1,984쌍은 공정값 36개도 수치 정밀도 허용 범위에서 모두 근접합니다. 물리적 샷 ID는 미확인이지만 독립 평가를 위해 이 후보를 함께 관리해야 합니다.

`ERR_FACT_QTY`는 집계 수입니다. 행별 불량 정답이 아니며 미래 검사 결과일 수 있으므로 예측 입력에서 제외합니다.

In [ ]:
display(pd.DataFrame(audit["labeled_unlabeled_candidate_overlap"].items(), columns=["항목", "결과"]))
display(pd.DataFrame(audit["unlabeled_data.csv"]["ERR_FACT_QTY"].items(), columns=["항목", "결과"]))

## 7. 공정 열을 이해하기

표는 전체 48종 열을 설명합니다. 0으로 고정된 온도 채널을 실제 0℃라고 단정하지 않습니다. RPM와 배압의 최대/평균 관계는 단위·배율을 확인하기 전까지 물리적인 차이 변수로 해석하지 않습니다.

In [ ]:
dictionary = pd.read_csv(PROJECT / "docs/process_column_dictionary.csv")
with pd.option_context("display.max_rows", 60, "display.max_colwidth", 85):
    display(dictionary[["column", "stage", "meaning_ko", "unit_from_guide", "caution_ko"]])
sensors = pd.read_csv(RESULTS / "sensor_coverage.csv")
display(sensors[(sensors["file"] == "labeled_data.csv") & (sensors["EQUIP_CD"] == "S14") & (sensors["nunique"] <= 1)])

## 8. 가이드북을 적용할 때 수정할 점과 다음 작업

- **49쪽:** 양품/불량에 각각 `fit_transform`을 적용한 예제는 추론 시 정답을 알아야 변환을 고를 수 있습니다. 새 실험은 시간 분할 후 학습 구간으로 전처리를 학습하고 모든 평가 데이터에 같은 변환을 적용합니다.
- **50쪽:** 오토인코더는 Dense 층의 공정값 복원 모델입니다. 타임스탬프가 있다고 AE가 시계열 모델이 되는 것은 아닙니다.
- **55쪽:** 5시그마 예제 임계값이 실제 불량 탐지율을 보장하지 않습니다. 경보량·양품 오탐과 함께 별도 검증합니다.
- **83–84쪽:** 평가 표의 불량 수가 제공된 표준화 파일과 달라 기존 baseline 숫자와 직접 비교할 수 없습니다.

다음은 **시간축의 원단위 EDA → 생산 구간·검증 분할 설계 → 단순 이상 탐지 → 정상 중심 AE 및 동일 조건 지도학습 대조**입니다. 새 모델 학습은 이 노트북에 포함되지 않습니다. 이상 점수는 불량 확률이 아니고, 이상 기여 열은 원인 확정이 아닙니다.

자세한 근거는 [가이드북 검토 문서](../docs/guidebook_review.md), 이전 실험은 [아카이브](../archive/2026-09-29_tabular_v1/README.md)에서 확인합니다.